# Validating the Extraction Pipeline against the Gold Set

*Stage 3 of the court-complaint project: measuring how accurately the extraction pipeline (`02_Complaint_RAG_Extraction.ipynb`) records the research variables, by comparing its output with complaints coded by hand.*

**Gold set:** 21 randomly selected complaints (V01–V13, V15–V22), hand-coded from the PDFs (`gold_set_final.xlsx`). V14 was excluded because the complaint is handwritten. For 2 complaints (V04, V05) the selected file is not the complaint: they count for the sorting check, and the extraction is compared on the other 19. Defendant lists were derived with AI assistance and are left out of the reported results (`assisted: defendants` in each case's notes).

**Key variables:** the plaintiff's **age**, **gender** and **race**; the **type of location** where the encounter began (primary setting and setting detail); and whether it involved a **traffic stop**. The sorting stage is checked too: is each selected file the case's operative, complete complaint?

**Steps**
1. Check the gold set for coding inconsistencies.
2. Copy the gold complaints into their own folder (done by Step 6b of `01_Separate_Complaints.ipynb`).
3. Run the extraction notebook (`02_Complaint_RAG_Extraction.ipynb`, `RUN = "validation"`) on that folder.
4. Compare: key variables, all variables, every disagreement, and further metrics (F1, kappa, confusion matrix).
5. Sensitivity check: results with the assisted values included.
6. Error analysis: classify each disagreement.
7. Summary for the methods description.

**Rule for a fair test:** the gold complaints are a held-out test set. Don't change the extraction pipeline's prompts or rules based on what these results show; if you do, the complaints become development data, and a new test set is needed for reporting.

**Status:** version 1 validated and adjudicated; results and error analysis are in **Results: Version 1** at the end of the notebook.

## Setup

In [ ]:
!pip install -q pandas openpyxl

In [ ]:
import re
import unicodedata
from difflib import SequenceMatcher
from math import sqrt
from pathlib import Path

import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Settings

- `SORTED`: the sorting notebook's output folder (sorted complaints, manifest).
- `GOLD`: the hand-coded gold set.
- `VALIDATION_DIR`: where the gold complaints are copied, and where the extraction notebook writes its output for them.
- `RESULTS`: the comparison workbook written by this notebook.

In [ ]:
SORTED = Path("/content/drive/MyDrive/COMPLAINTS_SORTED")
GOLD = SORTED / "gold_set_final.xlsx"
VALIDATION_DIR = SORTED / "validation_complaints"
PIPELINE = VALIDATION_DIR / "library_analysis.xlsx"        # written by the extraction notebook
RESULTS = SORTED / "validation_results.xlsx"
RESULTS_ALL = SORTED / "validation_results_assisted_included.xlsx"

print("Gold set found:", GOLD.exists())
print("Pipeline output found:", PIPELINE.exists())

## Comparison Functions

How the comparison works:
- **Matching:** complaints by file name (gold `complaint_file` = pipeline `source_file`), with the case number as a fallback; plaintiffs and defendants by name similarity within each complaint.
- **Normalization:** "not stated", "not applicable", blank and setting **U** all count as *not stated*; text is compared without case, accents or punctuation; race is compared by category (e.g. "African American" = "Black", "Navajo" = Native American); dates as dates.
- **Metrics per variable:** agreement with a 95% Wilson confidence interval; for variables that can be not stated: **recall** (when the complaint states it, how often the pipeline gets it right), **precision** (when the pipeline reports a value, how often it is right), and counts of **missed**, **unsupported** and **wrong** values.
- **Assisted coding:** fields marked `assisted: <fields>` in a case's notes are left out (or included, for the sensitivity check).
- **Wrong documents:** complaints whose selected file is not the complaint (`correct_document` = no) count for the sorting check but are left out of the extraction comparison.
- **Traffic stop in complaints without an encounter:** when the case scope is *employment* or *other*, traffic stop counts as "no", for the hand coding and the pipeline output alike (the complaint describes no stop at all).

In [ ]:
import re
import unicodedata
from difflib import SequenceMatcher
from math import sqrt
from pathlib import Path

import pandas as pd

NS = "not stated"
MISSING = {"", "nan", "none", "nat", "not stated", "not applicable", "n/a", "na", "unknown", "<na>"}

# fields compared at the complaint level: (gold column, pipeline column, kind)
CASE_FIELDS = [
    ("case_scope", "case_scope", "category"),
    ("primary_setting", "setting", "category"),
    ("setting_detail", "setting_detail", "category"),
    ("all_settings", "all_settings", "set"),
    ("traffic_stop", "traffic_stop", "category"),
    ("pattern_alleged", "pattern_alleged", "category"),
    ("criminal_suspicion", "criminal_suspicion", "category"),   # compared only where the gold setting is C
    ("date_filed", "date_filed", "date"),
    ("document_title", "document_title", "text"),
]
RACE_GROUPS = [
    ("Black", r"\bblack\b|african"),
    ("Hispanic/Latino", r"hispanic|latin[oax]|chican|mexican|puerto ric|dominican|colombian|salvador|guatemal"),
    ("Asian", r"\basian\b|chinese|korean|vietnam|filipin|japanese|indian american|south asian"),
    ("White", r"(?<!non-)(?<!non )\bwhite\b|caucasian"),
    ("Native American", r"native american|american indian|indigenous|alaska native|navajo|tribe|tribal|lakota|"
                        r"cherokee|sioux|apache|ojibwe|chippewa"),
    ("Middle Eastern/North African", r"arab|middle eastern|north african|persian|iranian"),
    ("Person of color (unspecified)", r"person of color|people of color|non-?white|minorit"),
]


# ---- normalization ------------------------------------------------------------------------------------
def norm(v):
    """Lower-case text, no accents or extra spaces; missing / not stated / not applicable -> 'not stated'."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return NS
    s = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode()
    s = re.sub(r"[\s_]+", " ", s).strip().lower()
    return NS if s in MISSING else s


def norm_value(v, kind):
    s = norm(v)
    if s == NS:
        return NS
    if kind == "set":
        codes = sorted({p.strip().upper() for p in re.split(r"[;,]", s) if p.strip()} - {"U"})
        return "; ".join(codes) if codes else NS
    if kind == "date":
        try:
            return pd.to_datetime(s).date().isoformat()
        except Exception:
            return s
    if kind == "text":
        return re.sub(r"[^a-z0-9 ]", "", s)
    if kind == "number":
        try:
            return str(int(float(s)))
        except Exception:
            return s
    if s == "u":                                                      # setting U = unclear / not stated
        return NS
    return s.upper() if re.fullmatch(r"[a-z]\d?", s) else s          # setting codes: t1 -> T1


def race_group(v):
    """Coarse race/ethnicity category of a description, for comparing the gold text with the pipeline's text."""
    s = norm(v)
    if s == NS:
        return NS
    groups = [g for g, rx in RACE_GROUPS if re.search(rx, s)]
    if len(groups) > 1 and "Person of color (unspecified)" in groups:
        groups.remove("Person of color (unspecified)")
    return "; ".join(groups) if groups else s


def name_key(v):
    s = norm(v)
    s = re.sub(r"\b(officer|sgt|sergeant|det|detective|deputy|trooper|lt|capt|chief|police|mr|ms|mrs|dr|jr|sr|ii|iii)\b\.?",
               " ", s)
    return re.sub(r"[^a-z ]", " ", s).split()


def name_similarity(a, b):
    ka, kb = name_key(a), name_key(b)
    if not ka or not kb:
        return 0.0
    if ka[-1] == kb[-1] and len(ka[-1]) > 2:                          # same surname
        return max(0.9, SequenceMatcher(None, " ".join(ka), " ".join(kb)).ratio())
    return SequenceMatcher(None, " ".join(ka), " ".join(kb)).ratio()


def wilson(k, n, z=1.96):
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(c - h, 3), round(c + h, 3))


# ---- development / test split for the LLM judge -------------------------------------------------------
def make_dev_test_split(gold_path, n_dev=10, seed=7, out_csv=None):
    """Assign n_dev gold complaints to 'dev' (for designing the LLM judge) and the rest to 'test' (used once).
    Saved as a CSV next to the gold workbook; never overwritten, so the split stays fixed."""
    out_csv = Path(out_csv or Path(gold_path).with_name("gold_split.csv"))
    if out_csv.exists():
        print(f"Split already exists, kept as is: {out_csv}")
        return pd.read_csv(out_csv, dtype=str)
    cases = pd.read_excel(gold_path, sheet_name="Cases", dtype=str).fillna("")
    ids = cases.loc[cases["sample_id"].str.match(r"V\d+"), "sample_id"]
    dev = set(ids.sample(n=min(n_dev, len(ids)), random_state=seed))
    split = pd.DataFrame({"sample_id": ids, "split": ["dev" if i in dev else "test" for i in ids]})
    split.to_csv(out_csv, index=False)
    print(f"Split saved: {(split['split'] == 'dev').sum()} dev, {(split['split'] == 'test').sum()} test -> {out_csv}")
    return split


# ---- staging the gold complaints for the extraction pipeline -----------------------------------------
def copy_gold_complaints(gold_path, complaints_dir, out_dir):
    """Copy the sampled complaints (Cases sheet, complaint_file) into out_dir, so the extraction notebook's
    process_folder(out_dir) runs on exactly the validation set."""
    import shutil
    cases = pd.read_excel(gold_path, sheet_name="Cases", dtype=str).fillna("")
    files = cases.loc[cases["sample_id"].str.match(r"V\d+"), "complaint_file"]
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    copied, missing = 0, []
    for f in files:
        src = Path(complaints_dir) / f
        if src.exists():
            shutil.copy2(src, out_dir / f)
            copied += 1
        else:
            missing.append(f)
    print(f"Copied {copied} of {len(files)} complaints to {out_dir}")
    for f in missing:
        print("  ⚠ not found:", f)


# ---- assisted coding: fields to leave out per complaint ------------------------------------------------
# In the gold workbook's Cases notes, write e.g. "assisted: defendants, setting, pattern" to mark fields coded with
# outside help; "coded with assistance" (without a list) marks the whole complaint.
ASSIST_GROUPS = {
    "defendants": {"n_defendants", "defendant:type", "defendant:identified"},
    "plaintiffs": {"n_plaintiffs", "plaintiff:identified", "age", "gender", "gender_basis", "race"},
    "parties": {"n_defendants", "defendant:type", "defendant:identified",
                "n_plaintiffs", "plaintiff:identified", "age", "gender", "gender_basis", "race"},
    "setting": {"primary_setting", "setting_detail", "all_settings", "traffic_stop"},
    "pattern": {"pattern_alleged"},
    "scope": {"case_scope"},
    "criminal suspicion": {"criminal_suspicion"},
    "race": {"race"}, "gender": {"gender", "gender_basis"}, "age": {"age"},
    "title": {"document_title"}, "date": {"date_filed"},
    "case name": set(), "case number": set(),
}
ALL_FIELDS = set().union(*ASSIST_GROUPS.values()) | {"case_scope", "document_title", "date_filed"}


def assisted_fields(note):
    """Fields marked as coded with assistance in a Cases note."""
    note = str(note or "").lower()
    if "coded with assistance" in note and "assisted:" not in note:
        return set(ALL_FIELDS)
    m = re.search(r"assisted:\s*([^.;\n]*)", note)
    if not m:
        return set()
    fields = set()
    for part in re.split(r",|\band\b", m.group(1)):
        part = part.strip()
        for key, group in ASSIST_GROUPS.items():
            if part and (part == key or part.rstrip("s") == key.rstrip("s")):
                fields |= group
        if part in ALL_FIELDS:
            fields.add(part)
    return fields


# ---- comparison ---------------------------------------------------------------------------------------
def _metrics(pairs):
    """pairs: list of (gold, pipeline) normalized values. Returns agreement and stated/not-stated metrics."""
    n = len(pairs)
    agree = sum(g == p for g, p in pairs)
    gold_stated = [(g, p) for g, p in pairs if g != NS]
    pipe_stated = [(g, p) for g, p in pairs if p != NS]
    missed = sum(p == NS for g, p in gold_stated)
    unsupported = sum(g == NS for g, p in pipe_stated)
    wrong = sum(g != NS and p != NS and g != p for g, p in pairs)
    lo, hi = wilson(agree, n)
    rec = sum(g == p for g, p in gold_stated) / len(gold_stated) if gold_stated else None
    prec = sum(g == p for g, p in pipe_stated) / len(pipe_stated) if pipe_stated else None
    return {"n": n, "agree": agree, "agreement": round(agree / n, 3) if n else None, "ci_low": lo, "ci_high": hi,
            "gold_stated": len(gold_stated), "recall_stated": None if rec is None else round(rec, 3),
            "pipeline_stated": len(pipe_stated), "precision_stated": None if prec is None else round(prec, 3),
            "missed (gold stated, pipeline not)": missed, "unsupported (pipeline stated, gold not)": unsupported,
            "wrong value": wrong}


def _evidence_lookup(evidence):
    """(source_file, field) -> (page, quote) from the pipeline's evidence table."""
    if evidence is None or evidence.empty:
        return {}
    ev = evidence.copy()
    return {(r["source_file"], r["field"]): (r.get("page"), r.get("quote")) for _, r in ev.iterrows()}


def compare(gold_path, pipeline_path, out_path, split_csv=None, use_split=None, assisted="exclude",
            name_threshold=0.75):
    """assisted="exclude": leave out fields marked as assisted in the notes (see assisted_fields);
    assisted="include": compare everything (to show whether the assistance changes the results)."""
    gold = pd.read_excel(gold_path, sheet_name=None, dtype=str)
    pipe = pd.read_excel(pipeline_path, sheet_name=None)
    gc = gold["Cases"].fillna("")
    gc = gc[gc["sample_id"].str.match(r"V\d+")]
    coded = gc.drop(columns=["sample_id", "case_number", "case_name", "case_name_source", "date_filed",
                             "complaint_file", "pages", "coder"], errors="ignore")
    gc = gc[coded.apply(lambda r: any(str(v).strip() for v in r), axis=1)]       # only complaints already coded
    skip = {}                                                         # sample_id -> assisted fields left out
    if assisted == "exclude":
        skip = {r["sample_id"]: assisted_fields(r.get("notes")) for _, r in gc.iterrows()}
    excluded = [f"{k}: {', '.join(sorted(v))}" for k, v in skip.items() if v]
    if split_csv and use_split:
        split = pd.read_csv(split_csv, dtype=str)
        gc = gc[gc["sample_id"].isin(split.loc[split["split"] == use_split, "sample_id"])]

    # sorting validation: is the selected file the case's operative, complete complaint?
    sort_vals = gc["correct_document"].map(norm) if "correct_document" in gc else pd.Series(dtype=str)
    sort_vals = sort_vals[sort_vals != NS]
    wrong_doc = gc.loc[gc["correct_document"].map(norm).str.startswith("no"), "sample_id"].tolist() \
        if "correct_document" in gc else []
    gc = gc[~gc["sample_id"].isin(wrong_doc)]                         # extraction can't be judged on a wrong file

    pc = pipe["cases"].copy()
    pp = pipe.get("plaintiffs", pd.DataFrame())
    pdf = pipe.get("defendants", pd.DataFrame())
    ev = _evidence_lookup(pipe.get("evidence"))

    def base(num):
        m = re.search(r"\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6}", str(num), re.I)
        return m.group(0).lower() if m else norm(num)

    # match complaints
    match_rows, sid_to_src = [], {}
    for _, g in gc.iterrows():
        src = g["complaint_file"] if g["complaint_file"] in set(pc["source_file"]) else None
        if src is None and g.get("case_number"):
            hit = pc[pc["case_number"].map(base) == base(g["case_number"])]
            src = hit["source_file"].iloc[0] if len(hit) == 1 else None
        sid_to_src[g["sample_id"]] = src
        match_rows.append({"sample_id": g["sample_id"], "complaint_file": g["complaint_file"],
                           "found_in_pipeline": src is not None, "pipeline_source_file": src})
    matched = gc[gc["sample_id"].map(lambda s: sid_to_src.get(s) is not None)]
    pc_idx = pc.set_index("source_file")

    summary, disagreements, all_pairs = [], [], []

    def record(level, field, pairs, details):
        if pairs:
            summary.append({"level": level, "field": field, **_metrics(pairs)})
        for d in details:
            all_pairs.append({"level": level, "field": field, "sample_id": d["sample_id"],
                              "gold": d["gold"], "pipeline": d["pipeline"]})
            if d["gold"] != d["pipeline"]:
                disagreements.append({"level": level, "field": field, **d})

    # complaint-level fields
    for gcol, pcol, kind in CASE_FIELDS:
        pairs, details = [], []
        for _, g in matched.iterrows():
            src = sid_to_src[g["sample_id"]]
            if gcol in skip.get(g["sample_id"], ()):
                continue
            if gcol == "criminal_suspicion" and norm(g.get("primary_setting")) != "c":
                continue
            gv = norm_value(g.get(gcol), kind)
            pv = norm_value(pc_idx.loc[src, pcol] if pcol in pc_idx.columns else None, kind)
            if gcol == "traffic_stop":
                # definition: a complaint without an encounter (scope employment or other) had no traffic stop;
                # applied in the same way to the hand coding and to the pipeline output
                if norm(g.get("case_scope")) in ("employment", "other"):
                    gv = "no"
                if "case_scope" in pc_idx.columns and norm(pc_idx.loc[src, "case_scope"]) in ("employment", "other"):
                    pv = "no"
            if gcol == "date_filed" and gv == NS:
                continue                                                    # not coded
            pairs.append((gv, pv))
            page, quote = ev.get((src, {"setting": "primary_setting", "setting_detail": "primary_setting_detail"}
                                  .get(pcol, pcol)), (None, None))
            details.append({"sample_id": g["sample_id"], "gold": gv, "pipeline": pv,
                            "pipeline_page": page, "pipeline_quote": quote})
        record("complaint", gcol, pairs, details)

    # counts of parties
    gp = gold.get("Plaintiffs", pd.DataFrame()).fillna("")
    gp = gp[gp["sample_id"].isin(matched["sample_id"])]
    gd = gold.get("Defendants", pd.DataFrame()).fillna("")
    gd = gd[gd["sample_id"].isin(matched["sample_id"])]
    for field, gtab, ptab in [("n_plaintiffs", gp, pp), ("n_defendants", gd, pdf)]:
        pairs, details = [], []
        for _, g in matched.iterrows():
            src = sid_to_src[g["sample_id"]]
            gv = norm_value(g.get(field), "number")
            if gv == NS or field in skip.get(g["sample_id"], ()):
                continue
            pv = str(int((ptab["source_file"] == src).sum())) if len(ptab) else "0"
            pairs.append((gv, pv))
            details.append({"sample_id": g["sample_id"], "gold": gv, "pipeline": pv,
                            "pipeline_page": None, "pipeline_quote": None})
        record("complaint", field, pairs, details)

    # plaintiffs: match by name within each complaint
    p_name = "name_clean" if "name_clean" in pp else "name"
    p_age = "age_num" if "age_num" in pp else "age"
    p_gender = "gender_std" if "gender_std" in pp else "gender"
    p_rows, field_pairs = [], {k: ([], []) for k in ["age", "gender", "gender_basis", "race"]}
    for sid, g_rows in gp.groupby("sample_id"):
        if "plaintiff:identified" in skip.get(sid, ()):
            continue
        src = sid_to_src[sid]
        cands = pp[pp["source_file"] == src].copy() if len(pp) else pd.DataFrame()
        used = set()
        for _, g in g_rows.iterrows():
            best, score = None, 0.0
            for j, c in cands.iterrows():
                s = name_similarity(g["name"], c.get(p_name))
                if j not in used and s > score:
                    best, score = j, s
            hit = best is not None and score >= name_threshold
            if hit:
                used.add(best)
            c = cands.loc[best] if hit else None
            row = {"sample_id": sid, "gold_name": g["name"], "pipeline_name": c.get(p_name) if hit else None,
                   "match": "found" if hit else "missed", "similarity": round(score, 2)}
            if hit:
                idx = int(c["idx"]) if "idx" in c and pd.notna(c["idx"]) else None
                vals = {
                    "age": (norm_value(g.get("age"), "number"), norm_value(c.get(p_age), "number")),
                    "gender": (norm(g.get("gender")), norm(c.get(p_gender))),
                    "gender_basis": (norm(g.get("gender_basis")), norm(c.get("gender_basis"))),
                    "race": (race_group(g.get("race_as_written")), race_group(c.get("race"))),
                }
                for k, (gv, pv) in vals.items():
                    row[f"gold_{k}"], row[f"pipeline_{k}"] = gv, pv
                    if k in skip.get(sid, ()):
                        continue
                    if k == "gender_basis" and gv == NS and pv == NS:
                        continue
                    page, quote = ev.get((src, f"plaintiffs[{idx}].{k.split('_')[0]}"), (None, None))
                    field_pairs[k][0].append((gv, pv))
                    field_pairs[k][1].append({"sample_id": sid, "gold": gv, "pipeline": pv, "pipeline_page": page,
                                              "pipeline_quote": quote, "plaintiff": g["name"]})
            p_rows.append(row)
        for j, c in cands.iterrows():
            if j not in used:
                p_rows.append({"sample_id": sid, "gold_name": None, "pipeline_name": c.get(p_name), "match": "extra"})
    for k, (pairs, details) in field_pairs.items():
        record("plaintiff", k, pairs, details)

    # defendants: match by name within each complaint
    d_name = "name_clean" if "name_clean" in pdf else "name"
    d_rows, type_pairs, type_details = [], [], []
    for sid, g_rows in gd.groupby("sample_id"):
        if "defendant:identified" in skip.get(sid, ()):
            continue
        src = sid_to_src[sid]
        cands = pdf[pdf["source_file"] == src] if len(pdf) else pd.DataFrame()
        used = set()
        for _, g in g_rows.iterrows():
            best, score = None, 0.0
            for j, c in cands.iterrows():
                s = name_similarity(g["name"], c.get(d_name))
                if j not in used and s > score:
                    best, score = j, s
            hit = best is not None and score >= name_threshold
            if hit:
                used.add(best)
                gv, pv = norm(g.get("type")), norm(cands.loc[best].get("type"))
                type_pairs.append((gv, pv))
                type_details.append({"sample_id": sid, "gold": gv, "pipeline": pv, "pipeline_page": None,
                                     "pipeline_quote": None, "defendant": g["name"]})
            d_rows.append({"sample_id": sid, "gold_name": g["name"],
                           "pipeline_name": cands.loc[best].get(d_name) if hit else None,
                           "match": "found" if hit else "missed", "similarity": round(score, 2)})
        for j, c in cands.iterrows():
            if j not in used:
                d_rows.append({"sample_id": sid, "gold_name": None, "pipeline_name": c.get(d_name), "match": "extra"})
    record("defendant", "type", type_pairs, type_details)

    # party-level recall and precision
    for level, rows in [("plaintiff", p_rows), ("defendant", d_rows)]:
        found = sum(r["match"] == "found" for r in rows)
        missed = sum(r["match"] == "missed" for r in rows)
        extra = sum(r["match"] == "extra" for r in rows)
        if found + missed:
            lo, hi = wilson(found, found + missed)
            summary.append({"level": level, "field": "identified (recall of gold parties)", "n": found + missed,
                            "agree": found, "agreement": round(found / (found + missed), 3), "ci_low": lo, "ci_high": hi})
        if found + extra:
            summary.append({"level": level, "field": "precision (pipeline parties that match gold)",
                            "n": found + extra, "agree": found, "agreement": round(found / (found + extra), 3),
                            "ci_low": wilson(found, found + extra)[0], "ci_high": wilson(found, found + extra)[1]})

    if len(sort_vals):
        ok = int((sort_vals == "yes").sum())
        lo, hi = wilson(ok, len(sort_vals))
        summary.insert(0, {"level": "sorting", "field": "correct_document (operative, complete complaint)",
                           "n": len(sort_vals), "agree": ok, "agreement": round(ok / len(sort_vals), 3),
                           "ci_low": lo, "ci_high": hi})
    summary_df = pd.DataFrame(summary)
    # headline table: the research variables that matter most for this project
    key = [("plaintiff", "age"), ("plaintiff", "gender"), ("plaintiff", "race"),
           ("complaint", "primary_setting"), ("complaint", "setting_detail"), ("complaint", "traffic_stop"),
           ("sorting", "correct_document (operative, complete complaint)")]
    headline = pd.concat([summary_df[(summary_df["level"] == l) & (summary_df["field"] == f)] for l, f in key
                          if len(summary_df)], ignore_index=True) if len(summary_df) else summary_df
    with pd.ExcelWriter(out_path, engine="openpyxl") as w:
        info = pd.DataFrame({
            "item": ["gold workbook", "pipeline workbook", "complaints compared", "complaints not found in pipeline",
                     "assisted fields left out", "split used", "not compared (wrong document selected)"],
            "value": [Path(gold_path).name, Path(pipeline_path).name, len(matched), len(gc) - len(matched),
                      ("; ".join(excluded) or "none") if assisted == "exclude" else "none (assisted values included)",
                      use_split or "all coded complaints", ", ".join(wrong_doc) or "none"]})
        info.to_excel(w, sheet_name="About", index=False)
        headline.to_excel(w, sheet_name="Key variables", index=False)
        summary_df.to_excel(w, sheet_name="Summary", index=False)
        pd.DataFrame(disagreements).to_excel(w, sheet_name="Disagreements", index=False)
        pd.DataFrame(all_pairs).to_excel(w, sheet_name="Pairs", index=False)
        pd.DataFrame(p_rows).to_excel(w, sheet_name="Plaintiffs", index=False)
        pd.DataFrame(d_rows).to_excel(w, sheet_name="Defendants", index=False)
        pd.DataFrame(match_rows).to_excel(w, sheet_name="Complaints", index=False)
        for sh in w.book.worksheets:
            for col in sh.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in col[:60])
                sh.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)
            sh.freeze_panes = "A2"
    print(f"Compared {len(matched)} complaints ({len(gc) - len(matched)} not found in the pipeline output) -> {out_path}")
    if excluded and assisted == "exclude":
        print("Assisted fields left out:", "; ".join(excluded))
    cols = ["level", "field", "n", "agreement", "ci_low", "ci_high", "recall_stated", "precision_stated",
            "missed (gold stated, pipeline not)", "unsupported (pipeline stated, gold not)", "wrong value"]
    return headline[[c for c in cols if c in headline]]

## Step 1: Check the Gold Set

Coding slips (a value outside the codebook, two codes in one setting cell, a count that doesn't match the rows) would show up as false errors in the comparison. Fix any issues listed below in the workbook, save it, and re-run this cell until the list is empty or every remaining item is a deliberate decision.

In [ ]:
# ---- Checking the gold set before the comparison ----------------------------------------------------
CODEBOOK_VALUES = {
    "correct_document": {"yes", "no: other document", "no: wrong version", "no: incomplete pages", "no"},
    "case_scope": {"encounter", "employment", "other", "not stated"},
    "primary_setting": {"T", "P", "J", "C", "R", "I", "S", "U"},
    "traffic_stop": {"yes", "no", "not stated"},
    "pattern_alleged": {"yes", "not stated"},
    "criminal_suspicion": {"present", "absent", "not stated", "not applicable"},
}
LEVEL2 = {"T1", "T2", "T3", "T4", "P1", "P2", "P3", "J1", "J2", "J3", "C1", "C2", "C3", "C4",
          "R1", "R2", "R3", "I1", "I2", "I3", "I4"}


def check_gold(gold_path):
    """List inconsistencies in the hand-coded gold set (one row per issue). Returns a DataFrame."""
    import datetime
    g = pd.read_excel(gold_path, sheet_name=None)
    cases = g["Cases"].copy()
    cases = cases[cases["sample_id"].astype(str).str.match(r"V\d+")]
    coded = cases.drop(columns=[c for c in ["sample_id", "case_number", "case_name", "case_name_source", "date_filed",
                                            "complaint_file", "pages", "coder", "notes", "check"] if c in cases])
    cases = cases[coded.apply(lambda r: r.notna().any(), axis=1)]
    pl = g["Plaintiffs"][g["Plaintiffs"]["sample_id"].isin(cases["sample_id"])]
    df = g["Defendants"][g["Defendants"]["sample_id"].isin(cases["sample_id"])]
    issues = []

    def add(sid, field, value, problem):
        issues.append({"sample_id": sid, "field": field, "value": value, "problem": problem})

    def blank_or_str(v):
        return "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v).strip()

    case_no = {s: blank_or_str(v) for s, v in zip(cases["sample_id"], cases["case_number"])}
    for _, r in cases.iterrows():
        sid = r["sample_id"]
        for col, allowed in CODEBOOK_VALUES.items():
            v = r.get(col)
            if pd.notna(v) and str(v).strip() and str(v).strip() not in allowed:
                add(sid, col, v, "not a codebook value")
        prim, det = str(r.get("primary_setting") or "").strip(), str(r.get("setting_detail") or "").strip()
        scope = norm(r.get("case_scope"))
        if scope == "encounter" and prim in ("", "nan", "U"):
            add(sid, "primary_setting", prim, "encounter without a setting")
        if det not in ("", "nan", "not stated"):
            if det not in LEVEL2:
                add(sid, "setting_detail", det, "one Level-2 code expected")
            elif prim and det[0] != prim:
                add(sid, "setting_detail", det, f"does not belong to primary setting {prim}")
        allset = {p.strip().upper()[:1] for p in re.split(r"[;,]", str(r.get("all_settings") or "")) if p.strip()}
        if prim in LEVEL2 or (prim and prim not in ("U", "nan") and allset and prim not in allset):
            add(sid, "all_settings", r.get("all_settings"), "should include the primary setting")
        if "S" in allset and scope == "encounter":
            add(sid, "all_settings", r.get("all_settings"), "S is for complaints without an encounter")
        cs = norm(r.get("criminal_suspicion"))
        if prim == "C" and cs == "not stated" and str(r.get("criminal_suspicion")).strip() == "not applicable":
            add(sid, "criminal_suspicion", "not applicable", "setting C: present, absent or not stated")
        if prim not in ("C", "", "nan") and cs not in ("not stated",):
            add(sid, "criminal_suspicion", r.get("criminal_suspicion"), "only for setting C (otherwise not applicable)")
        if norm(r.get("traffic_stop")) == "yes" and prim not in ("T",) and "T" not in allset:
            add(sid, "traffic_stop", "yes", "traffic stop but no vehicle setting (T)")
        for col in ["setting_page", "pattern_page"]:
            if isinstance(r.get(col), (datetime.datetime, datetime.date, pd.Timestamp)):
                add(sid, col, r.get(col), "page number turned into a date by the spreadsheet")
        for col, tab in [("n_plaintiffs", pl), ("n_defendants", df)]:
            n_rows = int((tab["sample_id"] == sid).sum())
            try:
                n = int(float(r.get(col)))
            except (TypeError, ValueError):
                add(sid, col, r.get(col), f"missing (the sheet has {n_rows} rows)")
                continue
            if n != n_rows:
                add(sid, col, n, f"differs from the {n_rows} rows on the sheet")
    for _, r in pl.iterrows():
        sid = r["sample_id"]
        if blank_or_str(r.get("case_number")) != case_no.get(sid):
            add(sid, "Plaintiffs.case_number", r.get("case_number"), "differs from the Cases sheet")
        age = norm(r.get("age"))
        if age != NS and not re.fullmatch(r"\d{1,3}", age):
            add(sid, "age", r.get("age"), "a number, or blank / not stated")
        g_ = norm(r.get("gender"))
        if g_ != NS and norm(r.get("gender_basis")) == NS:
            add(sid, "gender_basis", None, "gender coded without a basis (explicit / inferred)")
    for _, r in df.iterrows():
        if blank_or_str(r.get("case_number")) != case_no.get(r["sample_id"]):
            add(r["sample_id"], "Defendants.case_number", r.get("case_number"), "differs from the Cases sheet")
    out = pd.DataFrame(issues, columns=["sample_id", "field", "value", "problem"])
    print(f"{len(cases)} coded complaints checked: {len(out)} issue(s)")
    return out

In [ ]:
issues = check_gold(GOLD)
display(issues)

## Step 2: Copy the Gold Complaints (done in `01_Separate_Complaints.ipynb`, Step 6b: skip)

Step 6b of the sorting notebook updates the gold set's file names after re-sorting and copies the gold complaints into `validation_complaints/` (complaints whose selected file is not the complaint go to `_not_compared/`). The cell below does a simpler copy and is only needed if you have not run Step 6b.

In [ ]:
copy_gold_complaints(GOLD, SORTED / "complaints", VALIDATION_DIR)

## Step 3: Run the Extraction Pipeline (in the extraction notebook)

In `02_Complaint_RAG_Extraction.ipynb`, with `RUN = "validation"` in its Settings: install cell → Runtime → Restart session → click the next code cell → **Run cell and below**. It processes the complaints in `validation_complaints/` and writes `library_analysis.xlsx` there. Runs are resumable: if some complaints failed, run it again and it processes only those.

Use the pipeline exactly as it is now: no changes based on the gold set. Then check that its output is complete:

In [ ]:
print("Pipeline output found:", PIPELINE.exists())
if PIPELINE.exists():
    extracted = set(pd.read_excel(PIPELINE, sheet_name="cases")["source_file"])
    to_compare = sorted(p.name for p in VALIDATION_DIR.glob("*.pdf"))
    missing = [f for f in to_compare if f not in extracted]
    print(f"Complaints extracted: {len(extracted)} | in validation_complaints/: {len(to_compare)} | missing: {missing or 'none'}")

## Step 4: Compare

The **Key variables** table is the main result. The workbook also contains the full Summary, every Disagreement (with the pipeline's quote and page), plaintiff and defendant matching, and which complaints were compared.

Small sample: with about 20 complaints, confidence intervals are wide (e.g. 18 of 20 correct → about 70–97%). Report setting accuracy at Level 1 (`primary_setting`) first; Level-2 codes occur only once or twice each.

In [ ]:
key = compare(GOLD, PIPELINE, RESULTS)
display(key)

In [ ]:
summary = pd.read_excel(RESULTS, sheet_name="Summary")
display(summary)

### Step 4b: Further Metrics for the Key Variables

- **F1:** the harmonic mean of recall and precision of stated values, a single summary per variable.
- **Cohen's kappa:** agreement corrected for chance. It matters for skewed variables: if nearly all complaints have no traffic stop, always answering "no" gives high agreement but a kappa near 0. It is also the statistic used for inter-coder reliability, so pipeline–human and human–human agreement are directly comparable.
- **Confusion matrix** of the primary setting (rows: gold, columns: pipeline): which location types are confused with which.
- **Traffic stop:** sensitivity (real traffic stops identified) and specificity (encounters without a traffic stop correctly marked "no"); "not stated" from the pipeline counts as not identifying a stop.

With about 20 complaints, these are descriptive: kappa in particular is unstable at this sample size.

In [ ]:
def cohen_kappa(gold, pred):
    """Cohen's kappa for two lists of category labels."""
    labels = sorted(set(gold) | set(pred))
    n = len(gold)
    if n == 0:
        return None
    po = sum(g == p for g, p in zip(gold, pred)) / n
    pe = sum((gold.count(l) / n) * (pred.count(l) / n) for l in labels)
    return round((po - pe) / (1 - pe), 3) if pe < 1 else None


pairs = pd.read_excel(RESULTS, sheet_name="Pairs").fillna("not stated")
summary = pd.read_excel(RESULTS, sheet_name="Summary")
KEY = [("plaintiff", "age"), ("plaintiff", "gender"), ("plaintiff", "race"),
       ("complaint", "primary_setting"), ("complaint", "setting_detail"), ("complaint", "traffic_stop")]
rows = []
for level, field in KEY:
    p_ = pairs[(pairs["level"] == level) & (pairs["field"] == field)]
    s_ = summary[(summary["level"] == level) & (summary["field"] == field)]
    if p_.empty:
        continue
    rec = s_["recall_stated"].iloc[0] if len(s_) else None
    prec = s_["precision_stated"].iloc[0] if len(s_) else None
    f1 = round(2 * rec * prec / (rec + prec), 3) if rec and prec else None
    rows.append({"level": level, "field": field, "n": len(p_), "recall": rec, "precision": prec, "F1": f1,
                 "kappa": cohen_kappa(list(p_["gold"].astype(str)), list(p_["pipeline"].astype(str)))})
metrics = pd.DataFrame(rows)
display(metrics)

ps = pairs[(pairs["level"] == "complaint") & (pairs["field"] == "primary_setting")]
if len(ps):
    print("Primary setting: rows = gold, columns = pipeline")
    display(pd.crosstab(ps["gold"], ps["pipeline"], margins=True))

ts = pairs[(pairs["level"] == "complaint") & (pairs["field"] == "traffic_stop")]
if len(ts):
    yes, no = ts[ts["gold"] == "yes"], ts[ts["gold"] == "no"]
    sens = f"{(yes['pipeline'] == 'yes').sum()} of {len(yes)}" if len(yes) else "no gold traffic stops"
    spec = f"{(no['pipeline'] == 'no').sum()} of {len(no)}" if len(no) else "no gold 'no' values"
    print(f"Traffic stop: sensitivity {sens}; specificity {spec}")

with pd.ExcelWriter(RESULTS, mode="a", engine="openpyxl", if_sheet_exists="replace") as w:
    metrics.to_excel(w, sheet_name="Key metrics", index=False)
print("Saved as sheet 'Key metrics' in", RESULTS.name)

## Step 5: Sensitivity Check

The same comparison with the assisted values (defendant lists) included. If the results barely change, the AI assistance in coding did not shape the findings.

In [ ]:
key_all = compare(GOLD, PIPELINE, RESULTS_ALL, assisted="include")
both = pd.merge(key[["level", "field", "n", "agreement"]], key_all[["level", "field", "n", "agreement"]],
                on=["level", "field"], how="outer", suffixes=(" (assisted excluded)", " (assisted included)"))
display(both)
full = pd.read_excel(RESULTS_ALL, sheet_name="Summary")
display(full[full["level"] == "defendant"])

## Step 6: Error Analysis

Every disagreement on a key variable, with the pipeline's quote and page. For each one, decide its type in the `error_type` column of the saved file:

| error_type | Meaning |
|---|---|
| `pipeline` | the pipeline is wrong: it missed a stated value, misread it, or attributed someone else's attribute to the plaintiff |
| `gold` | the hand coding is wrong: correct the gold set (and note it) |
| `ambiguous` | the complaint genuinely allows both readings |
| `definition` | the codebook rule is unclear (e.g. which setting came first): refine the rule |

Then group the pipeline errors by cause (retrieval missed the passage, wrong person, inferred instead of explicit, wrong setting level, …). That grouping, not the overall percentage, shows what to improve.

**After classifying:** correct every `gold` row in `gold_set_final.xlsx`, with a note *"Corrected after adjudication: …"* in the case's notes, so the corrections are documented; then re-run Step 4 (and 4b, 5) with the corrected gold set. Use the same few `cause` labels throughout, so they can be counted.

In [ ]:
KEY_FIELDS = {"age", "gender", "race", "primary_setting", "setting_detail", "traffic_stop"}
dis = pd.read_excel(RESULTS, sheet_name="Disagreements")
key_dis = dis[dis["field"].isin(KEY_FIELDS)].copy()
key_dis["error_type"] = ""          # pipeline / gold / ambiguous / definition
key_dis["cause"] = ""               # e.g. missed passage, wrong person, inferred vs explicit
key_dis["comment"] = ""
review_path = SORTED / "validation_error_review.xlsx"
if review_path.exists():
    print("Review file already exists and was NOT overwritten:", review_path)
else:
    key_dis.to_excel(review_path, index=False)
    print(f"{len(key_dis)} disagreements on key variables -> {review_path}")
display(key_dis)

After filling in `error_type` (and `cause`) in `validation_error_review.xlsx`, summarize:

In [ ]:
rev = pd.read_excel(SORTED / "validation_error_review.xlsx").fillna("")
if (rev["error_type"] != "").any():
    display(pd.crosstab(rev["field"], rev["error_type"], margins=True))
    display(rev[rev["error_type"] == "pipeline"].groupby(["field", "cause"]).size().rename("n").reset_index())
else:
    print("No error types entered yet.")

## Step 7: Summary for the Methods Description

A first draft of the validation paragraph, filled in from the results. Edit it after the error analysis, in particular if some disagreements turn out to be gold-set errors (correct those, re-run Step 4, and regenerate).

In [ ]:
# ---- Step 7: Summary for the methods description ----
def pct(row):
    return f"{row['agreement']:.0%} (95% CI {row['ci_low']:.0%}–{row['ci_high']:.0%}, n = {int(row['n'])})"

k = pd.read_excel(RESULTS, sheet_name="Key variables").set_index(["level", "field"])     # all columns
about = pd.read_excel(RESULTS, sheet_name="About").set_index("item")["value"]

parts = []
for (level, field), label in [(("plaintiff", "age"), "age"),
                              (("plaintiff", "gender"), "gender"),
                              (("plaintiff", "race"), "race or ethnicity"),
                              (("complaint", "primary_setting"), "type of location (Level 1)"),
                              (("complaint", "setting_detail"), "specific location type (Level 2)"),
                              (("complaint", "traffic_stop"), "traffic stop")]:
    if (level, field) in k.index:
        parts.append(f"{label} {pct(k.loc[(level, field)])}")

text = (f"The extraction pipeline was validated against a gold set of {about.get('complaints compared')} randomly "
        f"selected complaints, coded by hand from the PDFs. Agreement with the hand coding was: "
        + "; ".join(parts) + ". ")

sort_key = ("sorting", "correct_document (operative, complete complaint)")
if sort_key in k.index:
    s = k.loc[sort_key]
    text += (f"The sorting stage selected the correct, complete complaint for {int(s['agree'])} of "
             f"{int(s['n'])} sampled cases. ")

review_path = SORTED / "validation_error_review.xlsx"
if review_path.exists():
    rev = pd.read_excel(review_path).fillna("")
    types = rev["error_type"].str.strip().str.lower().value_counts()
    text += (f"All {len(rev)} disagreements on the key variables were reviewed against the PDFs: "
             f"{types.get('gold', 0)} were errors in the hand coding (corrected before the final comparison), "
             f"{types.get('pipeline', 0)} pipeline errors, and {types.get('definition', 0)} differences in "
             f"coding rules. ")

text += ("Defendant lists, derived with AI assistance during coding, were excluded from the reported comparison. "
         "Given the small sample, confidence intervals are wide; results are a pilot validation.")
print(text)

## Results: Version 1 (Adjudicated)

*Pipeline version 1: gpt-4o, text-embedding-3-small, bge-reranker-base, 60 pages indexed. Gold set after adjudication. These figures are from the run described above; re-running Steps 4–7 updates the tables in `validation_results.xlsx`, not this text.*

| Variable | Agreement (95% CI) | n |
|---|---|---|
| Plaintiff age | **100%** (86–100%) | 23 plaintiffs |
| Plaintiff race or ethnicity | **96%** (79–99%) | 23 |
| Plaintiff gender | **91%** (73–98%) | 23 |
| Traffic stop | **79%** (57–92%) | 19 complaints |
| Type of location (Level 1) | **47%** (27–68%) | 19 |
| Specific location type (Level 2) | **47%** (27–68%) | 19 |
| Sorting: correct, complete complaint selected | **19 of 21** (90%) | 21 |

**Validation.** The extraction pipeline was validated against a gold set of complaints drawn at random from the sorted library and coded by hand from the PDFs. Of 21 sampled complaints, the sorting stage had selected the correct, complete complaint for 19; the extraction was compared on these 19 (23 plaintiffs). Agreement with the hand coding was 100% for plaintiff age (95% CI 86–100%), 96% for race or ethnicity (79–99%) and 91% for gender (73–98%); the pipeline reported no incorrect gender or age, and its only race error was an incomplete value. Agreement was lower for the encounter variables: 79% for whether a traffic stop occurred (57–92%) and 47% for the type of location where the encounter began (27–68%).

**Adjudication and error analysis.** All 34 disagreements on the key variables were reviewed against the PDFs. Eleven were errors in the hand coding, corrected before the final comparison; two reflected differences in coding rules, resolved by a common definition applied to both sources (a complaint without an encounter had no traffic stop); and 21 were pipeline errors.

| Cause of the pipeline errors | Errors |
|---|---|
| Setting check too strict: the correct passage was found, but the value was removed for lacking the expected cue words | 8 |
| Retrieval missed the passage | 5 |
| Wrong incident or passage (complaints describing several moments) | 4 |
| Follows from a missing setting (traffic stop) | 3 |
| Partial value (one of two stated races) | 1 |

The errors are concentrated in the location variables, and the most frequent cause is a design choice, not a misreading: a consistency check that discards correctly retrieved values. Gender errors were retrieval misses (the evidence was near the plaintiff's name but not among the retrieved passages), not wrong values.

Defendant lists, prepared with AI assistance during coding, were excluded from the reported comparison. With 19 complaints, confidence intervals are wide, and the results should be read as a pilot validation.

## Next Steps

**1. Version 2: fix the location variables.** The error analysis points to three changes, in order of impact:
- keep a setting value whose quote passes the citation check even without cue words, marked *for review* instead of deleted (the 8 + 3 errors);
- ignore cue words inside street addresses (e.g. "West 36th Street" read as a street stop);
- add retrieval queries aimed at where the encounter began (e.g. "where was plaintiff when officers approached"), for the missed and wrong-passage errors.

**2. Test version 2 on new complaints.** This test set has been seen and adjudicated, so measuring a fix on it would overstate the gain. Code 10–15 further complaints from the sample (V23 onward) with the same codebook and rules, and compare versions 1 and 2 on them: a before-and-after on unseen data.

**3. Further validation:** a second coder for inter-coder reliability (Cohen's kappa, comparable with the pipeline's kappa in Step 4b); a model comparison on the gold set (e.g. gpt-4o vs. gpt-4o-mini: accuracy, cost, time); an LLM judge, designed on development complaints and validated against the gold set, to review the full library.

**4. The full library** (466 complaints) with the better version, on a GPU runtime, in resumable batches (`02_Complaint_RAG_Extraction.ipynb`, `RUN = "full"`).

**Remaining coding item (not a key variable):** the gold set's *pattern alleged* column has no "yes" values while the pipeline found seven; re-code it from the PDFs before reporting that variable.